# Black-Litterman Backtest Examples

Complete usage examples and test cases for the backtest toolkit

## Setup

First, import the toolkit from the main backtest module:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add notebook directory to path
notebook_dir = str(Path.cwd())
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

# Run the main backtest notebook to load all functions and classes
# This imports everything: Markowitz, BlackLitterman, RollingWindowBacktest, 
# PerformanceMetrics, run_experiment, etc.
exec(open('backtest.ipynb').read().replace('get_ipython()', 'None'))

# Or alternatively, if you prefer using nbformat:
# import nbformat
# with open('backtest.ipynb') as f:
#     nb = nbformat.read(f, as_version=4)
#     code = '\n'.join(cell['source'] for cell in nb.cells if cell['cell_type'] == 'code')
#     exec(code)

## 1. Load Data

In [9]:
# Load your returns data
returns = pd.read_csv('etf_returns.csv', index_col=0, parse_dates=True)
print(f"Data shape: {returns.shape}")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")

Data shape: (2764, 10)
Date range: 2015-01-05 00:00:00 to 2025-12-30 00:00:00


## 2. Example 1: Test ONLY Improved Σ (Covariance)

In [11]:
# Simulate an improved covariance matrix (e.g., using shrinkage estimator)
# For demo, we'll use a slightly modified sample covariance
train_data_demo = returns.iloc[252*5:252*6]  # One year of training data
standard_sigma = train_data_demo.cov().values
improved_sigma_demo = standard_sigma * 0.95  # Simulate improvement (shrinkage)

# Simplest way: use dedicated function
comparison_sigma, results_sigma = run_experiment_improve_sigma_only(
    returns,
    improved_sigma=improved_sigma_demo,
    tau=0.05,
    train_window=252*3,  # 3 years for faster testing
    rebalance_freq=252,
    plot=True
)

print("\nSummary - Effect of Improved Σ Only:")
print(f"Sharpe Ratio: Standard BL = {comparison_sigma.loc['Sharpe Ratio', 'Standard BL']:.4f}, " 
      f"Improved = {comparison_sigma.loc['Sharpe Ratio', 'Improved BL']:.4f}")

NameError: name 'run_experiment_improve_sigma_only' is not defined

## 3. Example 2: Test ONLY Improved Ω (View Uncertainty)

In [ ]:
# Simulate an improved view uncertainty matrix (e.g., more confident views)
# Get standard omega first for comparison
train_data_demo = returns.iloc[252*5:252*6]
P_demo, q_demo, _ = get_standard_views(train_data_demo)
n_assets = len(returns.columns)
view_variance = np.array([np.dot(P_demo[i], np.dot(train_data_demo.cov().values, P_demo[i])) 
                          for i in range(len(P_demo))])
standard_omega_demo = np.diag(view_variance * 0.05)
improved_omega_demo = standard_omega_demo * 0.7  # Lower uncertainty = more confident views

# Simplest way: use dedicated function
comparison_omega, results_omega = run_experiment_improve_omega_only(
    returns,
    improved_omega=improved_omega_demo,
    tau=0.05,
    train_window=252*3,  # 3 years for faster testing
    rebalance_freq=252,
    plot=True
)

print("\nSummary - Effect of Improved Ω Only:")
print(f"Sharpe Ratio: Standard BL = {comparison_omega.loc['Sharpe Ratio', 'Standard BL']:.4f}, " 
      f"Improved = {comparison_omega.loc['Sharpe Ratio', 'Improved BL']:.4f}")

## 4. Example 3: Improve Both Σ and Ω

In [ ]:
improved_sigma = standard_sigma * 0.95
improved_omega = standard_omega_demo * 0.7

comparison, results = run_experiment(
    returns,
    custom_sigma=improved_sigma,
    custom_omega=improved_omega,
    tau=0.05,
    train_window=252*3,
    rebalance_freq=252,
    plot=True
)

print("\nImprovement Summary:")
print(f"Annual Return Improvement: {comparison.loc['Annual Return', 'Improved BL'] - comparison.loc['Annual Return', 'Standard BL']:.4f}")
print(f"Sharpe Ratio Improvement: {comparison.loc['Sharpe Ratio', 'Improved BL'] - comparison.loc['Sharpe Ratio', 'Standard BL']:.4f}")
print(f"Max Drawdown Improvement: {comparison.loc['Max Drawdown', 'Improved BL'] - comparison.loc['Max Drawdown', 'Standard BL']:.4f}")

## 5. Manual Examples

### Test ONLY Improved Σ (Manual way)

In [ ]:
# You can also pass parameters manually if you need more control
improved_sigma = standard_sigma * 0.95

comparison, results = run_experiment(
    returns,
    custom_sigma=improved_sigma,  # Your improved Σ
    custom_P=None,                # Use standard momentum P
    custom_q=None,                # Use standard q
    custom_omega=None,            # Use standard omega (auto-calculate)
    plot=True
)

### Test ONLY Improved Ω (Manual way)

In [ ]:
improved_omega = standard_omega_demo * 0.7

comparison, results = run_experiment(
    returns,
    custom_sigma=None,            # Use standard sample covariance
    custom_P=None,                # Use standard momentum P
    custom_q=None,                # Use standard q
    custom_omega=improved_omega,  # Your improved Ω
    plot=True
)

### Access Raw Results

In [ ]:
comparison, results = run_experiment(returns, custom_sigma=improved_sigma)

# Access individual strategy results
markowitz_values = results['markowitz']['values']
standard_bl_returns = results['standard_bl']['returns']
improved_bl_weights = results['improved_bl']['weights']

# Calculate custom metrics
improved_bl_metrics = PerformanceMetrics(
    results['improved_bl']['values'],
    results['improved_bl']['returns'],
    weights_history=results['improved_bl']['weights']
).get_metrics()

print("Your Improved BL Strategy:")
for metric, value in improved_bl_metrics.items():
    print(f"  {metric}: {value:.4f}")

## 6. Tips & Tricks

### Tip 1: Use Dedicated Functions for Single-Parameter Tests

When testing only one improvement, use the dedicated functions for clarity:
- `run_experiment_improve_sigma_only()` - isolate Σ improvements
- `run_experiment_improve_omega_only()` - isolate Ω improvements

### Tip 2: Adjust Window Sizes for Faster Testing

Use smaller windows during development/testing:

```python
# Quick test (3 years training window)
comparison, results = run_experiment(
    returns,
    custom_sigma=improved_sigma,
    train_window=252*3,  # Instead of default 252*5
    plot=True
)
```

### Tip 3: Compare Results Across Multiple Tests

```python
# Run multiple tests
comp1, res1 = run_experiment_improve_sigma_only(returns, sigma1)
comp2, res2 = run_experiment_improve_sigma_only(returns, sigma2)
comp3, res3 = run_experiment_improve_sigma_only(returns, sigma3)

# Compare improvements
print("Sharpe Improvement Comparison:")
print(f"Sigma 1: {comp1.loc['Sharpe Ratio', 'Improved BL'] - comp1.loc['Sharpe Ratio', 'Standard BL']:.4f}")
print(f"Sigma 2: {comp2.loc['Sharpe Ratio', 'Improved BL'] - comp2.loc['Sharpe Ratio', 'Standard BL']:.4f}")
print(f"Sigma 3: {comp3.loc['Sharpe Ratio', 'Improved BL'] - comp3.loc['Sharpe Ratio', 'Standard BL']:.4f}")
```